**NLP Homework 2 (2026)**


*   By 成功大學 資管所 碩二 徐敏慧
*   學號: R76141138



# LSTM-arithmetic



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
! pip install seaborn
! pip install opencc
! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
import opencc
import os
from sklearn.model_selection import train_test_split

data_path = '/content/drive/MyDrive/NLP/HW2'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 16.7 MB/s  0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,(40*0)-18=,-18
1,6+4+31=,41
2,4*11+13=,57
3,44*39*26=,44616
4,(39*42)+27=,1665


In [ ]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [ ]:
char_to_id = {}
id_to_char = {}

# write your code here
# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id is to conver charactors to ids, while id_to_char is the opposite
char_to_id = {'<pad>': 0, '<eos>': 1}

all_chars = set(''.join(df_train['src']))

for char in sorted (all_chars):
  char_to_id[char] = len(char_to_id)

id_to_char = {i: char for char, i in char_to_id.items()}

vocab_size = len(char_to_id)
print('Vocab size{}'.format(vocab_size))

Vocab size18


# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [ ]:
# Write your code here

def to_char_ids(text):
    # Convert char to its id
    return [char_to_id[c] for c in text] + [char_to_id['<eos>']]

def to_label_ids(row):
    # The length of a question
    question_len = len(row['src']) - len(row['tgt'])
    #
    return ([char_to_id['<pad>']] * question_len
            + [char_to_id[c] for c in row['tgt']]
            + [char_to_id['<eos>']])

for df in [df_train, df_eval]:
    df['char_id_list'] = df['src'].apply(to_char_ids)
    df['label_id_list'] = df.apply(to_label_ids, axis=1)



df_train.head()

,src,tgt,len,char_id_list,label_id_list
0,(40*0)-18=-18,-18,13,"[2, 11, 7, 4, 7, 3, 6, 8, 15, 17, 6, 8, 15, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 8, 15, 1]"
1,6+4+31=41,41,9,"[13, 5, 11, 5, 10, 8, 17, 11, 8, 1]","[0, 0, 0, 0, 0, 0, 0, 11, 8, 1]"
2,4*11+13=57,57,10,"[11, 4, 8, 8, 5, 8, 10, 17, 12, 14, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 12, 14, 1]"
3,44*39*26=44616,44616,14,"[11, 11, 4, 10, 16, 4, 9, 13, 17, 11, 11, 13, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 11, 11, 13, 8, 13, 1]"
4,(39*42)+27=1665,1665,15,"[2, 10, 16, 4, 11, 9, 3, 5, 9, 14, 17, 8, 13, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 13, 13, 1..."


# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|

In [ ]:
# batch_size (v2) = 256
batch_size = 256
# epochs (v2) = 4
# epochs (v3) = 10
epochs = 10
embed_dim = 256
# hidden_dim (v2) = 512
hidden_dim = 512
# lr = 0.002 (v2)
lr = 0.002
grad_clip = 1

# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- The, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        # return the amount of data
        return len(self.sequences)
        # Write your code here

    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        # Write your code here
        x = self.sequences.iloc[index, 0][:-1] # char_id_list
        # Write your code here
        y = self.sequences.iloc[index, 1][1:] # label_id_list
        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])

    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [ ]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [ ]:
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(ds_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)


# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

**LSTM**

In [ ]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        # Write your code here
        # Design your own model, which should contain embedding layer, several hidden layers and a linear layer

        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                            embedding_dim=embed_dim,
                                            padding_idx=char_to_id['<pad>'])

        self.rnn_layer1 = torch.nn.LSTM(input_size=embed_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)

        self.rnn_layer2 = torch.nn.LSTM(input_size=hidden_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)

        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=hidden_dim),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=vocab_size))

    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)

    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)

        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)

        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)

        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)

        batch_x = self.linear(batch_x)

        return batch_x

    def generator(self, start_char, max_len=200):

        char_list = [char_to_id[c] for c in start_char]

        next_char = None

        while len(char_list) < max_len:
            # Write your code here
            # Pack the char_list to tensor
            x = torch.tensor([char_list], device=self.embedding.weight.device)

            # Input the tensor to the embedding layer, RNN layers, linear respectively
            x = self.embedding(x)
            y, _ = self.rnn_layer1(x)
            y, _ = self.rnn_layer2(y)
            y = self.linear(y)

            next_char = y[0, -1].argmax().item()

            # Use argmax function to get the next token prediction

            if next_char == char_to_id['<eos>']:
                break

            char_list.append(next_char)

        return [id_to_char[ch_id] for ch_id in char_list]

In [ ]:
torch.manual_seed(2)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim)

In [ ]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])
# Write your code here. Cross-entropy loss function. The loss function should ignore <pad>
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
# Write your code here. Use Adam or AdamW for Optimizer

In [ ]:
print(device)

cuda


# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `1000` batches, output the current loss to monitor whether it is converging.

In [ ]:
from tqdm import tqdm
from copy import deepcopy
model = model.to(device)
model.train()
i = 0
for epoch in range(1, epochs+1):
    # The process bar
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        # Write your code here
        # Clear the gradient
        optimizer.zero_grad()


        batch_pred_y = model(batch_x.to(device), batch_x_lens)

        # Write your code here
        # Input the prediction and ground truths to loss function
        loss = criterion(batch_pred_y.reshape(-1, vocab_size), batch_y.to(device).reshape(-1))
        # Back propagation
        loss.backward()

        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) # gradient clipping

        # Write your code here
        # Optimize parameters in the model
        optimizer.step()

        i+=1
        if i%50==0:
            bar.set_postfix(loss = loss.item())

    # Evaluate your model
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        batch_x = row['src']
        batch_y = row['tgt']

        question = batch_x.split('=')[0] + '='

        with torch.no_grad():
          prediction = model.generator(question)


        prediction = ''.join(prediction)[len(question):]
        if prediction == batch_y:
          matched += 1
        total += 1
        # An example of using generator: model.generator('1+1=')
        # Write your code here. Input the batch_x to the model and generate the predictions
        # Write your code here.
        # Check whether the prediction match the ground truths
        # Compute exact match (EM) on the eval dataset
        # EM = correct/total


    print(matched/total)
    #torch.save(model.state_dict(), '/content/drive/MyDrive/NLP/HW2/checkpoint.pt')
    #torch.save(model.state_dict(), '/content/drive/MyDrive/NLP/HW2/checkpoint_v2.pt')
    torch.save(model.state_dict(), '/content/drive/MyDrive/NLP/HW2/checkpoint_v3.pt')

Train epoch 1: 100%|██████████| 3907/3907 [02:12<00:00, 29.43it/s, loss=0.402]
Validation epoch 1: 10000it [00:45, 218.43it/s]


0.5069


Train epoch 2: 100%|██████████| 3907/3907 [02:17<00:00, 28.40it/s, loss=0.304]
Validation epoch 2: 10000it [00:45, 222.05it/s]


0.627


Train epoch 3: 100%|██████████| 3907/3907 [02:15<00:00, 28.88it/s, loss=0.196]
Validation epoch 3: 10000it [00:45, 221.12it/s]


0.706


... (10 outputs left out)

Train epoch 9: 100%|██████████| 3907/3907 [01:59<00:00, 32.65it/s, loss=0.0916]
Validation epoch 9: 10000it [00:39, 250.15it/s]


0.8756


Train epoch 10: 100%|██████████| 3907/3907 [01:59<00:00, 32.76it/s, loss=0.0695]
Validation epoch 10: 10000it [00:40, 246.70it/s]


0.8772


Running environment: `Colab`

Python version: `python 3.13`

***

##### 1. Describe your model architecture, data preprocessing methods, and hyperparameter settings, and justify your choices.  (5%)

Answer:

**Model Architecture:**

- Embedding (18 -> 256, padding_idx = `<pad>`)
- LSTM layer 1 (256 -> 512)
- LSTM layer 2 (512 -> 512)
- Linear (512 -> 512) -> ReLU -> Linear (512 -> 18)
- Loss: CrossEntropyLoss with `ignore_index = <pad>`
- Optimizer: Adam



**Data Preprocessing Methods:**

1.   Character-level tokenization: The vocabulary contains 18 tokens in total, including the digits (`0-9`), the symbols (`+, -, *, (), =`), and the two tokens (`<pad>`, `<eos>`).
2.   Concatenation of question and answer: Each sample becomes one sequence. `<eos>` is attached to the end of each sequence so that the model knows when to stop generating.
3.   Labels: Every position of the question part is replaced with `<pad>`, so that the loss is computed only on the answers and `<eos>`.
4.   Shifting for teaching forcing: The input drops the last token and the targer drops the first token.
5.   Padding: Sequences in a batch are padded with `<pad>` to the same length.


`Hyperparameter Settings:`

| Version | batch_size | epochs | embed_dim | hidden_dim | lr | grad_clip | Accuracy (EM) |
|---|---|---|---|---|---|---|---|
| v1 | 64 | 2 | 256 | 256 | 0.001 | 1 | 54.0% |
| v2 | 256 | 4 | 256 | 512 | 0.002 | 1 | 77.6% |
| **v3 (final)** | 256 | 10 | 256 | 512 | 0.002 | 1 | **87.7%** |


`Why choose this particular model architecture? Why choose this hyperparameter settings?`

- LSTM: In a controlled small-scale comparison (See Question 2 for further details), LSTM reached 24.8%, compared with 17.1% for GRU and 13.2% for RNN under identical settings.
- v1 to v2: I increased `hidden_dim` from 256 to 512 to give the model more memory capacity, for multiplication requires keeping track of many digits. I increased `batch_size` from 64 to 256 to use the GPU more efficiently as well. And I also raised `lr` from 0.001 to 0.002 to compensate for fewer parameter updates per epoch. Under such a setting, accuracy progressed from 54.0% to 77.6%.
- v2 to v3: In v2, accuracy was still rising by about 8 to 10 points per epoch at epoch 4 (aka the last epoch in v2), so I supposed that the model had not converged. For this I dial up the number of epochs to 10. The improvement became small at the end (87.56% to 87.72% between the last two epochs), which may suggest that the model was close to convergence.

***

##### 2. If you use 2-layer LSTM or GRU instead of RNN, what will happen to the quality of your answer generation? Why? (10%)

Answer:

**Setting:** For fairness, all three models were trained on the same random 100,000-sample subset of the training set (`random_state = 0`) for 2 epochs, with batch_size 64, embed_dim 256, hidden_dim 256, and lr 0.001. Only the recurrent layer type differed. Other than that, the settings are highly identical.
Evaluation used the full evaluation set.

| Model (2 layers) | Epoch 1 | Epoch 2 |
|---|---|---|
| RNN | 12.4% | 13.2% |
| GRU | 9.2% | 17.1% |
| LSTM | 8.5% | **24.8%** |

**Observations:**
- RNN had the highest accuracy after epoch 1 but almost stopped improving in epoch 2.
- LSTM started lowest among the three, yet it improved the most after epoch 2 and achieved the best result among the three.
- GRU fell between the two. The improvements from epoch 1 to epoch 2 are also notably better than RNN.

**Why:**
- I suppose that the slow progress of RNN comes from the fact that it updates its hidden state by mixing the old state with each new input, which may make the information from early tokens fade quickly. Also, during training, gradients are multiplied repeatedly through time and shrink, which may in turn makes long-range dependencies hard to grasp. The RNN can quickly learn simple patterns, but then plateaus in this very case.
- LSTM has a separate cell state and three gates (input, forget, output) that control what to store, discard, and output. Such an architecture lets digits be preserved across many time steps and lets gradients flow more easily, which may be the reason why it seems to grasp harder problems such as multiplication better.
- GRU is a simplified version with two gates and no separate cell state. It has fewer parameters and less control over memory than LSTM, which is consistent with its results.

***

##### 3. If we construct an training set using three-digit numbers while the evaluation set is constructed from two-digit numbers, what will happen to the quality of your answer generation? (10%)

Answer:


**Setting:**
- I generated 100,000 training expressions in which every number is a three-digit number (`100–999`), using the same four formats as the original data (`A op B`, `A op B op C`, `(A op B) op C`, `A op (B op C)`), with answers (computed using Python).
- For evaluation, only the 5,033 questions in the original evaluation set are retained, where every number has exactly two digits.
- The model was a 2-layer LSTM with the same small-scale settings as Question 2.

**Result:**
- Exact-match accuracy was 0.0% on the 5,033 two-digit evaluation questions after epoch 1 and epoch 2.
- For reference, accuracy on 500 newly generated three-digit questions (the same distribution as training) was only 0.2%, so the model had not learned exact arithmetic even this case. That said, the types of errors are very different.

**Examples: two-digit questions (out of distribution)**

| Question | Correct | Model |
|---|---|---|
| `34+47-32=` | 49 | 691 |
| `(47-27)+26=` | 46 | 601 |
| `(46*38)+25=` | 1773 | 171113 |
| `(49*41)-46=` | 1963 | 191244 |
| `13*(46+12)=` | 754 | 80000 |
| `47*18*34=` | 28764 | 2511124 |

**Examples: three-digit questions (in distribution)**

| Question | Correct | Model |
|---|---|---|
| `893+625-765=` | 753 | 761 |
| `(186-556)-639=` | -1009 | -1001 |
| `(282+234)+960=` | 1476 | 1486 |
| `758*624*322=` | 152303424 | 152244444 |

**Observations and Analysis (Assumptions):**
- In distribution, the model's answers have the correct number of digits and correct leading digits, and are often off by less than 10 in value. The model has likely learned to estimate the answer but not to compute every digit exactly.
- Out of distribution, the leading digit is often still roughly right, yet the answers are consistently too long, usually by about two digits. Even simple additions such as `34+47-32` (answer: 49) receive three-digit answers. I suppose this has something to do with the fact that during training, every operand had three digits, and that the answers were always large. It's plausible that the model learned the typical length of an answer from the training distribution.
- After the first few digits, the model often fills the answer with repeated digits, which most likely indicates that it falls back on frequent patterns when uncertain.
- Conclusion: It seems that the model does not learn arithmetic rules that transfer to new number ranges. It learns statistical patterns tied to the training distribution, including the expected answer length. The model seems to lack the sense of scale under the shift of distribution.

***

##### 4. If we construct a training set that includes 20% incorrect answers, how will this affect the quality of the generated responses? Present some examples. (10%)

Answer:

**Setting:**
- This is conducted using the same 100,000-sample subset as Question 2.
-  I randomly selected 20% of the samples and replaced each answer with a wrong one by adding a random offset value between `−10` and `+10`. Note that the offset value is never 0.
- The evaluation set was left clean. The model was a 2-layer LSTM with the same settings as the clean LSTM baseline.

| Training data | Epoch 1 | Epoch 2 |
|---|---|---|
| Clean (baseline) | 8.5% | 24.8% |
| 20% incorrect answers | 7.8% | **9.4%** |

**Examples (model trained with 20% incorrect answers):**

| Question | Correct | Model |
|---|---|---|
| 34+47-32= | 49 | 49 |
| 33-41-15= | -23 | -25 |
| (47-27)+26= | 46 | 44 |
| 12-6-22= | -16 | -15 |
| 19*35+33= | 698 | 691 |
| (49*41)-46= | 1963 | 1974 |
| 5*29*19= | 2755 | 2716 |
| 47*18*34= | 28764 | 25146 |


**Observations:**

On the first 1,000 evaluation questions, the model answered 886 incorrectly. Of these wrong answers, 560 (63.2%) were within ±10 of the correct answerm meaning that more than half of the wrong answers are actually near-miss

**Analysis (& Assmuptions):**
- The model's answers almost always have the correct sign, number of digits, and even leading digits. However, the last digits are often wrong. In other words, the model produces near-miss answers.
- This mirrors how the noise was created, as incorrect labels were the true answer plus an offset of ±1 to ±10. Training on such labels teaches the model that answers near the truth are acceptable, which weakens its pressure to get every digit exactly right.
- The reduction in accuracy (from 24.8% to 9.4%) is likely attributed to how an answer is deemed correct: exact match requires every digit to be correct, hence these near misses all count as failures.
- Similar questions come with conflicting answers, which may make the training set pull the model in inconsistent directions and learning becomes less precise.

***

##### 5. Why do we need gradient clipping during training? (5%)

Answer:
- RNN-type models are trained with backpropagation through time. As the gradient flows back through each time step, it is multiplied by the same recurrent weight matrix again and again. If this matrix tends to enlarge values, the gradient can grow exponentially.
- A huge gradient causes a huge parameter update, which could destroy what the model has already learned.
- Gradient clipping helps limit the size of the gradient before each update. Take my code for example, I used `clip_grad_value_` with `grad_clip = 1`, which clamps every gradient element to the range [−1, 1].
- Sequences such as `(39*42)+27=1665` are around 15–20 characters long, and multiplication answers vary greatly in size, so gradients can change sharply. Clipping keeps training stable and prevents the sudden spikes in losses across different epochs.


***

##### … Anything that can strengthen your report. (5%)

Answer:

**1. Training curve of the final model (v3)**

| Epoch | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| Accuracy | 50.7% | 62.7% | 70.6% | 78.5% | 79.9% | 80.8% | 83.7% | 84.5% | 87.6% | 87.7% |
|Loss| 40.2% | 30.4% | 19.6% | 13.4% | 15.8% | 12% | 12.8% | 9.0% | 9.1% | 7.0% |

**Observations:**

- Accuracy rose quickly in the first four epochs and then improved more slowly. It approaches convergence by epoch 10. It is certainly possible that the accuracy can be further improved provided that the number of epochs is increased. However, considering the inflation in the execution time, the number of epochs in this experiment did not extend beyond 10.
- The training loss decreased from 0.40 to 0.07. Judging by how the loss pleateaued after the 6th epoch, I suppose it is rather safe to assume that the model is near convergence.



***
##### Generative AI Usage Statement

Answer:

I consulted Claude on several occassions, as listed below.

**Reports**
- Provided formatting assitance for all tables presented above.
- Proofread text for grammatical errors ( & for terminalogy misuse).

**Code Snippets**
- Assited in constructing the three-digit dataset.
- Assisted in constructing the corrupted dataset (the training set that includes 20% incorrect answers).


**P.S.:**

To avoid confusion, I renamed the checkpoint file from `checkpoint_v3.pt` to `checkpoint.pt`. So the attached pt. file is the final version of the model (the v3 model).